In [ ]:
from lightning_train import MyLightningModule
from argparse import Namespace
import torch
from torch.utils.data import Dataset, DataLoader

args = Namespace(
    model_name_or_path="intfloat/simlm-base-msmarco",
    q_max_len=32,
    p_max_len=144,
)
module = MyLightningModule(args)



/traindata/maksim/miniconda3/envs/lightning/lib/python3.11/site-packages/tqdm/auto.py:21: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
Some weights of BertModel were not initialized from the model checkpoint at intfloat/simlm-base-msmarco and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
from lightning_train import JsonlDataset, collate_fn
from lib import RestorableSampler
from transformers import AutoTokenizer
from functools import partial
from torch.utils.data import Dataset, DataLoader

train_dataset = JsonlDataset("data/train.jsonl")
tokenizer = AutoTokenizer.from_pretrained("intfloat/simlm-base-msmarco")
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=RestorableSampler(train_dataset, module.consumed_samples),
    collate_fn=partial(collate_fn, tokenizer, args),
    num_workers=4,
    pin_memory=True,
    drop_last=True,
)

Loading data: 400782it [00:11, 34666.96it/s]


In [7]:
examples = []
for i, example in enumerate(train_loader):
    examples.append(example)
    if i > 10:
        break
examples[0]

{'query': {'input_ids': tensor([[  101,  2003,  9016,  1037,  3696,  1997, 27395,  4668,   102,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0],
         [  101,  2054,  2024,  1996,  5918,  2005,  8452,  5427,  8292,   102,
              0,     0,     0,     0,     0,     0,     0,     0,     0],
         [  101,  2054,  2003,  1996,  3296, 10300,  1997,  1037,  2557, 27179,
          21416, 10727,   102,     0,     0,     0,     0,     0,     0],
         [  101,  2054,  2003,  3642,  4205,   102,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0],
         [  101,  2073,  2003,  2019, 20668,  2912,  9565, 26985,  3364,  2013,
            102,     0,     0,     0,     0,     0,     0,     0,     0],
         [  101,  2054,  2003,  1037,  3650,  2386,  1999,  1996,  3212,   102,
              0,     0,     0,     0,     0,     0,     0,     0,     0],
         [  101,  2054,  2515,  2516,  5427,  4047,  

In [10]:
examples[0]["query"]["input_ids"]

tensor([[  101,  2003,  9016,  1037,  3696,  1997, 27395,  4668,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2054,  2024,  1996,  5918,  2005,  8452,  5427,  8292,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2054,  2003,  1996,  3296, 10300,  1997,  1037,  2557, 27179,
         21416, 10727,   102,     0,     0,     0,     0,     0,     0],
        [  101,  2054,  2003,  3642,  4205,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2073,  2003,  2019, 20668,  2912,  9565, 26985,  3364,  2013,
           102,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2054,  2003,  1037,  3650,  2386,  1999,  1996,  3212,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2054,  2515,  2516,  5427,  4047,   102,     0,     0,     0,
         

In [13]:
for i in range(32):
    print(tokenizer.decode(examples[0]["query"]["input_ids"][i]))

[CLS] is fever a sign of allergic reaction [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] what are the requirements for delaware insurance ce [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] what is the annual salary of a radiologic technologist [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] what is code adam [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] where is anushka tollywood actor from [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] what is a corpsman in the navy [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] what does title insurance protect [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] what forms are needed to file an extension for il [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] do bitters contain alcohol [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] salvation army muskogee ok p

In [14]:
for i in range(32):
    print(tokenizer.decode(examples[1]["query"]["input_ids"][i]))

[CLS] amount of silver in half dollar [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] which credit cards does sam ' s club accept [SEP] [PAD] [PAD] [PAD] [PAD]
[CLS] how old is james worthy [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] where can i spend avios points [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] how to convert from millions to billions [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] what causes an inflamed pancreas [SEP] [PAD] [PAD] [PAD]
[CLS] what is the reset time on a 60 % duty cycle welder [SEP]
[CLS] what is a true pangram [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] what county is eufaula al in [SEP] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] mood stabilizers and side effects [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] average cost of extended warranty on used car [SEP] [PAD] [PAD] [PAD] [PAD]
[CLS] calories in arby ' s farmhouse turkey salad [SEP] [PAD] [PAD] [PAD]
[CLS] exogenous definition [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [

In [17]:
cp2 = torch.load("runs/train_v1/epoch=1-step=30.ckpt")
cp2["MyDataModule"]

/tmp/ipykernel_2235750/754658142.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cp2 = torch.load("runs/train_v1/epoch=1-step=30.ckpt")


{'train_sampler': {'indices': [291,
   2203,
   971,
   65,
   163,
   894,
   2250,
   4767,
   5205,
   6169,
   5859,
   4716,
   5313,
   3624,
   1439,
   3433,
   5325,
   5216,
   2107,
   4530,
   6352,
   6494,
   6621,
   2147,
   160,
   5825,
   2804,
   6355,
   6311,
   4096,
   2827,
   1104,
   5181,
   3412,
   5120,
   5416,
   5900,
   568,
   679,
   2241,
   6154,
   899,
   6106,
   6292,
   4165,
   1823,
   2049,
   3281,
   3787,
   3737,
   1918,
   2683,
   758,
   3233,
   4264,
   1126,
   5165,
   556,
   1396,
   6043,
   3089,
   3563,
   1164,
   6412,
   3904,
   4969,
   272,
   6257,
   729,
   434,
   2888,
   4928,
   5809,
   4621,
   5118,
   559,
   2692,
   5641,
   233,
   1472,
   560,
   1793,
   2478,
   6481,
   4940,
   5621,
   6974,
   3779,
   1685,
   4772,
   6212,
   3569,
   509,
   1923,
   3221,
   4603,
   549,
   4424,
   4192,
   5976,
   6934,
   4238,
   194,
   4869,
   1022,
   4789,
   5598,
   6141,
   653,
   2357,
   4

In [19]:
cp = torch.load("runs/train_v2/epoch=0-step=10.ckpt")
cp["MyDataModule"]

/tmp/ipykernel_2235750/305227199.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cp = torch.load("runs/train_v2/epoch=0-step=10.ckpt")


{'train_sampler': {'indices': [6798,
   4213,
   3903,
   83,
   1162,
   4884,
   1336,
   3179,
   5100,
   2545,
   2012,
   4476,
   1295,
   363,
   546,
   4682,
   557,
   6157,
   2216,
   2253,
   4182,
   2246,
   3603,
   4178,
   2872,
   3369,
   4334,
   2975,
   327,
   2315,
   6300,
   4060,
   6700,
   1525,
   769,
   4,
   4759,
   222,
   1223,
   2618,
   149,
   6316,
   5737,
   4108,
   6061,
   6611,
   5021,
   1855,
   5883,
   276,
   2371,
   365,
   724,
   175,
   4988,
   5859,
   1285,
   1906,
   4964,
   4907,
   3830,
   2690,
   5809,
   1433,
   6828,
   1505,
   4261,
   6861,
   2638,
   6855,
   1429,
   3151,
   2086,
   6343,
   6842,
   4972,
   6429,
   291,
   792,
   6749,
   3402,
   2656,
   3036,
   2024,
   2039,
   740,
   6209,
   1220,
   3300,
   4051,
   2367,
   2090,
   6206,
   388,
   942,
   6374,
   6391,
   5211,
   3730,
   2375,
   3329,
   5913,
   4245,
   4079,
   3836,
   4801,
   1577,
   6223,
   5775,
   5074,
   

{'train_sampler': {'indices': [4724,
   5892,
   6194,
   3289,
   6177,
   1041,
   3106,
   1870,
   6053,
   1262,
   1538,
   6440,
   5311,
   4416,
   2204,
   3629,
   5195,
   3726,
   744,
   5085,
   6287,
   4871,
   213,
   5292,
   4683,
   1446,
   6457,
   4366,
   6934,
   3630,
   1559,
   2090,
   3929,
   4572,
   3791,
   693,
   6523,
   3110,
   981,
   6911,
   3642,
   437,
   4123,
   4344,
   5033,
   1169,
   1606,
   2908,
   3803,
   1944,
   1128,
   916,
   1709,
   2383,
   6193,
   6935,
   3565,
   5785,
   6822,
   4931,
   6092,
   3670,
   3227,
   208,
   352,
   2417,
   960,
   3191,
   1177,
   1006,
   436,
   1776,
   6089,
   6084,
   3752,
   4669,
   6510,
   4414,
   5021,
   2342,
   6178,
   4662,
   6827,
   1793,
   2488,
   4788,
   2547,
   6709,
   2805,
   5279,
   149,
   3968,
   5460,
   5693,
   6198,
   4524,
   3477,
   2294,
   6455,
   6825,
   448,
   4395,
   3318,
   579,
   3695,
   6613,
   2510,
   1224,
   2953,
   4

In [8]:
len(cp["MyDataModule"]["train_sampler"]["indices"])

400782

In [8]:
for k, v in cp["state_dict"].items():
    print(k, v.shape)


model.model.embeddings.word_embeddings.weight torch.Size([30522, 768])
model.model.embeddings.position_embeddings.weight torch.Size([512, 768])
model.model.embeddings.token_type_embeddings.weight torch.Size([2, 768])
model.model.embeddings.LayerNorm.weight torch.Size([768])
model.model.embeddings.LayerNorm.bias torch.Size([768])
model.model.encoder.layer.0.attention.self.query.weight torch.Size([768, 768])
model.model.encoder.layer.0.attention.self.query.bias torch.Size([768])
model.model.encoder.layer.0.attention.self.key.weight torch.Size([768, 768])
model.model.encoder.layer.0.attention.self.key.bias torch.Size([768])
model.model.encoder.layer.0.attention.self.value.weight torch.Size([768, 768])
model.model.encoder.layer.0.attention.self.value.bias torch.Size([768])
model.model.encoder.layer.0.attention.output.dense.weight torch.Size([768, 768])
model.model.encoder.layer.0.attention.output.dense.bias torch.Size([768])
model.model.encoder.layer.0.attention.output.LayerNorm.weight tor

In [26]:
cp["lr_schedulers"]

[{'step_size': 1,
  'gamma': 0.9,
  'base_lrs': [2e-05],
  'last_epoch': 0,
  'verbose': False,
  '_step_count': 1,
  '_get_lr_called_within_step': False,
  '_last_lr': [2e-05]}]

In [1]:
from lightning_train import JsonlDataset
from torch.utils.data import DataLoader, RandomSampler

train_dataset = JsonlDataset("data/train.jsonl")


train_sampler = RandomSampler(train_dataset)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,  # You can adjust the batch size as needed
    sampler=train_sampler,
    num_workers=4,  # You can adjust the number of workers as needed
    pin_memory=True
)

train_dataset[0]


/traindata/maksim/miniconda3/envs/lightning/lib/python3.11/site-packages/tqdm/auto.py:21: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
Loading data: 400782it [00:13, 29590.66it/s]


{'query': ')what was the immediate impact of the success of the manhattan project?',
 'pos_doc': ['Introduction',
  'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.'],
 'neg_doc': [['-',
   'Uploaded on Sep 19, 2009. This video shows one way to manage calendar spread trade that is going in the wrong direction. This video highlights the use of the thinkorswim analyze tab to determine the impact of adjusting the trade. This video provided by http://www.success-with-options.com.'],
  ['The World Clock - Time Zone difference from USA â Kansas â Manhattan',
   'The World Clock-Time Zone difference from U.S.A. – Kansas – Manhattan.dd or subtract the given number of hours to/from Manhattan time to get the time in these 